# Prepare Embeddings for DB15K

This notebook generates text and visual embeddings from scraped data.

## Prerequisites

Run `scrape_db15k_data.py` first to collect raw data:

```bash
python scrape_db15k_data.py \
    --data_dir ../../MoMoK/datasets/DB15K \
    --output_dir ./scraped_data \
    --text_mode uri_only \
    --download_images \
    --image_sample 100
```

## What This Notebook Does

1. **Load scraped data** (triples, texts, images)
2. **Encode text** with Sentence-BERT (768-dim)
3. **Encode images** with ResNet50 (2048-dim) or CLIP (512-dim)
4. **Save embeddings** in GWM-RNN compatible format

## Requirements

```bash
pip install sentence-transformers torch torchvision pillow
```

## 1. Configuration

In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# PATHS
# ============================================================================

# Input: scraped data directory
SCRAPED_DATA_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\link-prediction\multimodal\db15k\scraped_data")

# Output: embeddings directory
OUTPUT_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\link-prediction\multimodal\db15k\processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'triples').mkdir(exist_ok=True)
(OUTPUT_DIR / 'embeddings').mkdir(exist_ok=True)

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================

# Text encoder
TEXT_MODEL = 'all-mpnet-base-v2'  # Options: all-mpnet-base-v2 (768-dim), all-MiniLM-L6-v2 (384-dim)

# Vision encoder
VISION_MODEL = 'resnet50'  # Options: resnet50, resnet101, clip

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print("="*70)
print("EMBEDDING GENERATION")
print("="*70)
print(f"Scraped data: {SCRAPED_DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Text model: {TEXT_MODEL}")
print(f"Vision model: {VISION_MODEL}")
print(f"Device: {DEVICE}")
print("="*70)

## 2. Load Scraped Data

In [ ]:
print("\n[1/6] Loading scraped data...")

# Load metadata
with open(SCRAPED_DATA_DIR / 'metadata.json', 'r') as f:
    metadata = json.load(f)

print(f"\nDataset: {metadata['dataset']}")
print(f"Entities: {metadata['num_entities']:,}")
print(f"Relations: {metadata['num_relations']:,}")
print(f"Images: {metadata['num_images']:,} ({metadata['image_coverage']*100:.1f}% coverage)")

# Load vocabularies
with open(SCRAPED_DATA_DIR / 'entity2id.json', 'r', encoding='utf-8') as f:
    entity2id = json.load(f)

with open(SCRAPED_DATA_DIR / 'relation2id.json', 'r', encoding='utf-8') as f:
    relation2id = json.load(f)

print(f"\n✓ Loaded vocabularies")

# Load texts
with open(SCRAPED_DATA_DIR / 'entity_texts.json', 'r', encoding='utf-8') as f:
    entity_texts = json.load(f)

with open(SCRAPED_DATA_DIR / 'relation_texts.json', 'r', encoding='utf-8') as f:
    relation_texts = json.load(f)

print(f"✓ Loaded texts")
print(f"  Entity texts: {len(entity_texts):,}")
print(f"  Relation texts: {len(relation_texts):,}")

# Load triples
def load_triples(file_path):
    triples = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            h, r, t = line.strip().split(' ', 2)
            h_id = entity2id[h]
            r_id = relation2id[r]
            t_id = entity2id[t]
            triples.append((h_id, r_id, t_id))
    return triples

train_triples = load_triples(SCRAPED_DATA_DIR / 'triples_train.txt')
valid_triples = load_triples(SCRAPED_DATA_DIR / 'triples_valid.txt')
test_triples = load_triples(SCRAPED_DATA_DIR / 'triples_test.txt')

train_triples_tensor = torch.tensor(train_triples, dtype=torch.long)
valid_triples_tensor = torch.tensor(valid_triples, dtype=torch.long)
test_triples_tensor = torch.tensor(test_triples, dtype=torch.long)

print(f"✓ Loaded triples")
print(f"  Train: {train_triples_tensor.shape}")
print(f"  Valid: {valid_triples_tensor.shape}")
print(f"  Test: {test_triples_tensor.shape}")

# Load image paths
with open(SCRAPED_DATA_DIR / 'image_paths.json', 'r') as f:
    image_paths_dict = json.load(f)

# Convert string keys to int
image_paths = {int(k): SCRAPED_DATA_DIR / v for k, v in image_paths_dict.items()}

print(f"✓ Loaded image paths: {len(image_paths):,}")

## 3. Encode Text with Sentence-BERT

In [ ]:
print("\n[2/6] Loading text encoder...")

try:
    from sentence_transformers import SentenceTransformer
    
    text_encoder = SentenceTransformer(TEXT_MODEL)
    text_encoder.to(DEVICE)
    
    text_dim = text_encoder.get_sentence_embedding_dimension()
    print(f"✓ Loaded {TEXT_MODEL}")
    print(f"  Dimension: {text_dim}")
    
    TEXT_ENCODER_AVAILABLE = True
except ImportError:
    print("❌ sentence-transformers not installed")
    print("   Install with: pip install sentence-transformers")
    TEXT_ENCODER_AVAILABLE = False

if TEXT_ENCODER_AVAILABLE:
    print("\n[3/6] Encoding texts...")
    
    # Encode entity texts
    print("  Encoding entity texts...")
    entity_text_embeddings = text_encoder.encode(
        entity_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_tensor=True,
        device=DEVICE
    )
    entity_text_embeddings = entity_text_embeddings.cpu()
    
    # Encode relation texts
    print("  Encoding relation texts...")
    relation_text_embeddings = text_encoder.encode(
        relation_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_tensor=True,
        device=DEVICE
    )
    relation_text_embeddings = relation_text_embeddings.cpu()
    
    print(f"\n✓ Entity text embeddings: {entity_text_embeddings.shape}")
    print(f"✓ Relation text embeddings: {relation_text_embeddings.shape}")
    
    # Verify normalization
    norms = torch.norm(entity_text_embeddings, p=2, dim=1)
    print(f"  Norm check: mean={norms.mean():.4f}, std={norms.std():.4f}")
else:
    print("\n[3/6] Skipping text encoding (sentence-transformers not available)")
    entity_text_embeddings = None
    relation_text_embeddings = None

## 4. Load Vision Encoder

In [ ]:
print("\n[4/6] Loading vision encoder...")

try:
    import torchvision.models as models
    import torchvision.transforms as transforms
    
    if VISION_MODEL == 'resnet50':
        resnet = models.resnet50(pretrained=True)
        vision_encoder = torch.nn.Sequential(*list(resnet.children())[:-1])
        vision_dim = 2048
        print(f"✓ Loaded ResNet50 (dimension: {vision_dim})")
        
    elif VISION_MODEL == 'resnet101':
        resnet = models.resnet101(pretrained=True)
        vision_encoder = torch.nn.Sequential(*list(resnet.children())[:-1])
        vision_dim = 2048
        print(f"✓ Loaded ResNet101 (dimension: {vision_dim})")
        
    elif VISION_MODEL == 'clip':
        try:
            import clip
            vision_encoder, img_transform = clip.load("ViT-B/32", device=DEVICE)
            vision_dim = 512
            print(f"✓ Loaded CLIP (dimension: {vision_dim})")
        except ImportError:
            print("❌ CLIP not available, falling back to ResNet50")
            resnet = models.resnet50(pretrained=True)
            vision_encoder = torch.nn.Sequential(*list(resnet.children())[:-1])
            vision_dim = 2048
    
    vision_encoder.to(DEVICE)
    vision_encoder.eval()
    
    # Image preprocessing
    if VISION_MODEL != 'clip':
        img_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])
    
    VISION_ENCODER_AVAILABLE = True
    
except ImportError:
    print("❌ torchvision not installed")
    print("   Install with: pip install torchvision")
    VISION_ENCODER_AVAILABLE = False
    vision_encoder = None
    img_transform = None
    vision_dim = None

## 5. Encode Images

In [ ]:
def encode_image(image_path, encoder, transform, device, vision_model='resnet50'):
    """Encode a single image to embedding."""
    try:
        img = Image.open(image_path).convert('RGB')
        
        if vision_model == 'clip':
            img_tensor = transform(img).unsqueeze(0).to(device)
            with torch.no_grad():
                features = encoder.encode_image(img_tensor)
        else:
            img_tensor = transform(img).unsqueeze(0).to(device)
            with torch.no_grad():
                features = encoder(img_tensor)
            features = features.squeeze()
        
        return features.cpu()
    except Exception as e:
        return None

if VISION_ENCODER_AVAILABLE and len(image_paths) > 0:
    print("\n[5/6] Encoding images...")
    
    num_entities = len(entity2id)
    entity_visual_embeddings = []
    entity_image_mask = []
    
    for entity_id in tqdm(range(num_entities), desc="  Processing entities"):
        if entity_id in image_paths:
            img_path = image_paths[entity_id]
            img_emb = encode_image(img_path, vision_encoder, img_transform, DEVICE, VISION_MODEL)
            
            if img_emb is not None:
                entity_visual_embeddings.append(img_emb)
                entity_image_mask.append(True)
            else:
                entity_visual_embeddings.append(torch.zeros(vision_dim))
                entity_image_mask.append(False)
        else:
            entity_visual_embeddings.append(torch.zeros(vision_dim))
            entity_image_mask.append(False)
    
    entity_visual_embeddings = torch.stack(entity_visual_embeddings)
    entity_image_mask = torch.tensor(entity_image_mask)
    
    # Normalize valid images
    if entity_image_mask.sum() > 0:
        entity_visual_embeddings[entity_image_mask] = torch.nn.functional.normalize(
            entity_visual_embeddings[entity_image_mask],
            p=2,
            dim=1
        )
    
    print(f"\n✓ Visual embeddings: {entity_visual_embeddings.shape}")
    print(f"  WITH images: {entity_image_mask.sum().item():,} ({entity_image_mask.float().mean()*100:.1f}%)")
    print(f"  WITHOUT images: {(~entity_image_mask).sum().item():,}")
    
    if entity_image_mask.sum() > 0:
        norms = torch.norm(entity_visual_embeddings[entity_image_mask], p=2, dim=1)
        print(f"  Norm check: mean={norms.mean():.4f}, std={norms.std():.4f}")
else:
    print("\n[5/6] Skipping image encoding (no images or encoder unavailable)")
    entity_visual_embeddings = None
    entity_image_mask = None

## 6. Save Embeddings

In [ ]:
print("\n[6/6] Saving embeddings...")

# Save triples
torch.save(train_triples_tensor, OUTPUT_DIR / 'triples' / 'train.pt')
torch.save(valid_triples_tensor, OUTPUT_DIR / 'triples' / 'valid.pt')
torch.save(test_triples_tensor, OUTPUT_DIR / 'triples' / 'test.pt')
print("  ✓ Saved triples")

# Save text embeddings
if entity_text_embeddings is not None:
    torch.save(entity_text_embeddings, OUTPUT_DIR / 'embeddings' / 'entity_text_sbert.pt')
    torch.save(relation_text_embeddings, OUTPUT_DIR / 'embeddings' / 'relation_text_sbert.pt')
    print(f"  ✓ Saved text embeddings")
    print(f"    entity_text_sbert.pt: {entity_text_embeddings.shape}")
    print(f"    relation_text_sbert.pt: {relation_text_embeddings.shape}")

# Save visual embeddings
if entity_visual_embeddings is not None:
    vision_name = VISION_MODEL if VISION_MODEL != 'clip' else 'clip'
    torch.save(entity_visual_embeddings, OUTPUT_DIR / 'embeddings' / f'entity_image_{vision_name}.pt')
    torch.save(entity_image_mask, OUTPUT_DIR / 'embeddings' / 'entity_image_mask.pt')
    print(f"  ✓ Saved visual embeddings")
    print(f"    entity_image_{vision_name}.pt: {entity_visual_embeddings.shape}")
    print(f"    entity_image_mask.pt: {entity_image_mask.shape}")

# Save vocabularies
with open(OUTPUT_DIR / 'entity2id.json', 'w', encoding='utf-8') as f:
    json.dump(entity2id, f, indent=2, ensure_ascii=False)

with open(OUTPUT_DIR / 'relation2id.json', 'w', encoding='utf-8') as f:
    json.dump(relation2id, f, indent=2, ensure_ascii=False)
print("  ✓ Saved vocabularies")

# Save final metadata
final_metadata = {
    'dataset_name': 'DB15K',
    'source': 'Custom embeddings from scraped data',
    'num_entities': len(entity2id),
    'num_relations': len(relation2id),
    'num_train_triples': len(train_triples),
    'num_valid_triples': len(valid_triples),
    'num_test_triples': len(test_triples),
    'text_model': TEXT_MODEL,
    'text_embedding_dim': int(entity_text_embeddings.shape[1]) if entity_text_embeddings is not None else None,
    'vision_model': VISION_MODEL,
    'image_embedding_dim': int(entity_visual_embeddings.shape[1]) if entity_visual_embeddings is not None else None,
    'image_availability_rate': float(entity_image_mask.float().mean()) if entity_image_mask is not None else 0.0,
    'entities_with_images': int(entity_image_mask.sum()) if entity_image_mask is not None else 0,
}

with open(OUTPUT_DIR / 'metadata.json', 'w') as f:
    json.dump(final_metadata, f, indent=2)
print("  ✓ Saved metadata")

print("\n" + "="*70)
print("✅ EMBEDDING GENERATION COMPLETE!")
print("="*70)
print(f"\nOutput: {OUTPUT_DIR}")
print("\nGenerated files:")
print("  triples/")
print("    train.pt, valid.pt, test.pt")
print("  embeddings/")
if entity_text_embeddings is not None:
    print("    entity_text_sbert.pt, relation_text_sbert.pt")
if entity_visual_embeddings is not None:
    print(f"    entity_image_{VISION_MODEL}.pt, entity_image_mask.pt")
print("  entity2id.json, relation2id.json")
print("  metadata.json")

## 7. Verification

In [ ]:
print("="*70)
print("VERIFICATION")
print("="*70)

def check_file(path, description):
    if path.exists():
        size_mb = path.stat().st_size / (1024**2)
        print(f"✓ {description:<40} ({size_mb:>6.2f} MB)")
        return True
    else:
        print(f"✗ {description:<40} MISSING")
        return False

print("\nChecking files:")
all_ok = True
all_ok &= check_file(OUTPUT_DIR / 'triples' / 'train.pt', 'triples/train.pt')
all_ok &= check_file(OUTPUT_DIR / 'triples' / 'valid.pt', 'triples/valid.pt')
all_ok &= check_file(OUTPUT_DIR / 'triples' / 'test.pt', 'triples/test.pt')

if entity_text_embeddings is not None:
    all_ok &= check_file(OUTPUT_DIR / 'embeddings' / 'entity_text_sbert.pt', 'entity_text_sbert.pt')
    all_ok &= check_file(OUTPUT_DIR / 'embeddings' / 'relation_text_sbert.pt', 'relation_text_sbert.pt')

if entity_visual_embeddings is not None:
    all_ok &= check_file(OUTPUT_DIR / 'embeddings' / f'entity_image_{VISION_MODEL}.pt', f'entity_image_{VISION_MODEL}.pt')
    all_ok &= check_file(OUTPUT_DIR / 'embeddings' / 'entity_image_mask.pt', 'entity_image_mask.pt')

all_ok &= check_file(OUTPUT_DIR / 'entity2id.json', 'entity2id.json')
all_ok &= check_file(OUTPUT_DIR / 'relation2id.json', 'relation2id.json')
all_ok &= check_file(OUTPUT_DIR / 'metadata.json', 'metadata.json')

if all_ok:
    print("\n✅ All files present!")
else:
    print("\n⚠️  Some files missing")

print("\n" + "="*70)
print("STATISTICS")
print("="*70)
print(f"\nEntities: {len(entity2id):,}")
print(f"Relations: {len(relation2id):,}")
print(f"\nTriples:")
print(f"  Train: {len(train_triples):,}")
print(f"  Valid: {len(valid_triples):,}")
print(f"  Test: {len(test_triples):,}")

if entity_text_embeddings is not None:
    print(f"\nText embeddings:")
    print(f"  Model: {TEXT_MODEL}")
    print(f"  Dimension: {entity_text_embeddings.shape[1]}")
    print(f"  Entities: {entity_text_embeddings.shape[0]:,}")
    print(f"  Relations: {relation_text_embeddings.shape[0]:,}")

if entity_visual_embeddings is not None:
    print(f"\nVisual embeddings:")
    print(f"  Model: {VISION_MODEL}")
    print(f"  Dimension: {entity_visual_embeddings.shape[1]}")
    print(f"  Entities: {entity_visual_embeddings.shape[0]:,}")
    print(f"  Coverage: {entity_image_mask.float().mean()*100:.1f}%")

print("\n✅ Ready for training!")

## Next Steps

Now that embeddings are ready, you can:

1. **Generate context embeddings**:
```bash
cd ../../link-prediction/multimodal
python generate_context_embeddings.py \
    --data_dir ./db15k/processed \
    --aggregation mean \
    --top_k 20
```

2. **Generate negative samples**:
```bash
python generate_negatives.py \
    --data_dir ./db15k/processed \
    --num_negatives 64 \
    --seed 42
```

3. **Train model**:
```bash
python train.py \
    --data_dir ./db15k/processed \
    --output_dir ../../../../trained/multimodal/db15k/exp1 \
    --text_embedding sbert \
    --image_embedding resnet50 \
    --epochs 100 \
    --batch_size 128
```

Good luck! 🚀